In [1]:
#Q1 - a

import numpy as np

# Load data
D = np.genfromtxt("lines.csv", delimiter=",", skip_header=1)

x1 = D[:, 0]
y1 = D[:, 3]

points = np.column_stack((x1, y1))
centroid = np.mean(points, axis=0)
Xc = points - centroid

U, S, Vt = np.linalg.svd(Xc)

direction = Vt[0]

normal = Vt[1]

a, b = normal
c = -a*centroid[0] - b*centroid[1]

print("TLS Line parameters:", a, b, c)

TLS Line parameters: 0.7735616496467872 -0.6337210539312553 -3.794192210845812


In [2]:
#Q1 - b

import random

X_cols = D[:, :3]
Y_cols = D[:, 3:]

X_all = X_cols.flatten()
Y_all = Y_cols.flatten()

points = np.column_stack((X_all, Y_all))

def fit_line(p1, p2):
    a = p2[1] - p1[1]
    b = p1[0] - p2[0]
    c = -(a*p1[0] + b*p1[1])
    return a, b, c

def distance(a, b, c, point):
    x, y = point
    return abs(a*x + b*y + c) / np.sqrt(a*a + b*b)

def ransac(points, iterations=1000, threshold=0.5):
    best_inliers = []

    for _ in range(iterations):
        p1, p2 = random.sample(list(points), 2)
        a, b, c = fit_line(p1, p2)

        inliers = []
        for p in points:
            if distance(a, b, c, p) < threshold:
                inliers.append(p)

        if len(inliers) > len(best_inliers):
            best_inliers = inliers

    return np.array(best_inliers)

# Find 3 lines
remaining = points.copy()
lines = []

for i in range(3):
    inliers = ransac(remaining)
    lines.append(inliers)

    # remove inliers
    remaining = np.array([p for p in remaining if not any((p == inliers).all(1))])

In [3]:
#Q2 - Part 1: Click two points

import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib tk

img = cv2.imread("earrings.jpg")
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

points = []

fig, ax = plt.subplots()
ax.imshow(img_rgb)
ax.set_title("Click two points on the earring")

def onclick(event):
    if event.xdata is not None and len(points) < 2:
        points.append((event.xdata, event.ydata))
        ax.plot(event.xdata, event.ydata, 'ro', markersize=5)
        fig.canvas.draw()
        if len(points) == 2:
            plt.close()
            print("Done! Points collected:", points)

fig.canvas.mpl_connect('button_press_event', onclick)
plt.show(block=True)

2026-04-30 13:50:44.608 python[85616:88731729] +[IMKClient subclass]: chose IMKClient_Legacy
2026-04-30 13:50:44.608 python[85616:88731729] +[IMKInputSession subclass]: chose IMKInputSession_Legacy


Done! Points collected: [(np.float64(98.6861471861472), np.float64(518.7034632034631)), (np.float64(478.2532467532468), np.float64(507.621212121212))]


In [4]:
#Q2 - Part 2: Compute pixel length

p1, p2 = points
pixel_length = np.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)
print("Pixel length:", pixel_length)


Pixel length: 379.72884978999235


In [8]:
#Q3 - a

import cv2 as cv
import numpy as np

N = 6
global n
n = 0
p1 = np . empty (( N ,2) )
p2 = np . empty (( N, 2))

DISPLAY_W , DISPLAY_H = 900 , 650

# mouse callback function
def draw_circle ( event ,x ,y , flags , param ) :
    global n
    p = param[0]
    if event == cv.EVENT_LBUTTONDOWN :
        cv.circle(param[1] ,( x , y ) ,5 ,(255 ,0 ,0) , -1)
        p[n] = (x , y)
        n += 1

im1 = cv.imread('c1.jpg', cv.IMREAD_REDUCED_COLOR_4)
im2 = cv.imread('c2.jpg', cv.IMREAD_REDUCED_COLOR_4)

im1copy = im1.copy()
im2copy = im2.copy()

cv.namedWindow('Image 1', cv.WINDOW_AUTOSIZE)

param = [p1, im1copy]
cv.setMouseCallback('Image 1', draw_circle, param)

while(1) :
    cv.imshow('Image 1', im1copy)
    if n == N:
        break
    if cv.waitKey(20) & 0xFF == 27:
        break

cv.destroyWindow('Image 1')

param = [p2 , im2copy]
n = 0

cv.namedWindow("Image 2", cv.WINDOW_AUTOSIZE)
cv.setMouseCallback('Image 2', draw_circle, param)

while(1) :
    cv.imshow('Image 2', im2copy)
    if n == N:
        break
    if cv.waitKey(20) & 0xFF == 27:
        break

cv.destroyWindow('Image 2')

print(p1)
print(p2)

#Convert to float32
p1 = p1.astype(np.float32)
p2 = p2.astype(np.float32)

#Compute homography
H, _ = cv.findHomography(p1, p2)

#Warp image
warped = cv.warpPerspective(im1, H, (im2.shape[1], im2.shape[0]))

cv.imwrite("results/warp_manual.jpg", warped)

cv.imshow("Warped Image", warped)
print("Press any key to exit...")
cv.waitKey(0)
cv.destroyAllWindows()

[[244.  51.]
 [508. 150.]
 [324. 640.]
 [268. 620.]
 [249. 630.]
 [ 75. 563.]]
[[ 19. 446.]
 [321.  21.]
 [485. 194.]
 [323. 502.]
 [177. 576.]
 [ 97. 520.]]
Press any key to exit...


In [9]:
#Q3 - b

diff = cv.absdiff(im2, warped)
cv.imwrite("results/diff_manual.jpg", diff)

cv.imshow("Difference", diff)
cv.waitKey(0)
cv.destroyAllWindows()

In [8]:
#Q3 - c

import cv2

im1 = cv2.imread("c1.jpg")
im2 = cv2.imread("c2.jpg")

#ORB detector
orb = cv2.ORB_create()

kp1, des1 = orb.detectAndCompute(im1, None)
kp2, des2 = orb.detectAndCompute(im2, None)

bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(des1, des2)

#Sort matches
matches = sorted(matches, key=lambda x: x.distance)

#Draw matches
img_matches = cv2.drawMatches(im1, kp1, im2, kp2, matches[:50], None)

cv2.imwrite("results/matches.jpg", img_matches)

True

In [ ]:
#Q3 - d

import numpy as np

#Extract matched points
src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1,1,2)
dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1,1,2)

#Compute homography using RANSAC
H, _ = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC)

#Warp image
warped = cv2.warpPerspective(im1, H, (im2.shape[1], im2.shape[0]))

#Difference image
diff = cv2.absdiff(im2, warped)

cv2.imwrite("results/warp_auto.jpg", warped)
cv2.imwrite("results/diff_auto.jpg", diff)

True